# Experimentación con canary 🐤

In [17]:
import torch

print(f"Is CUDA available? {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    MODEL = "nvidia/canary-1b-flash"
else:
    MODEL = "nvidia/canary-1b-v2"

gpu_count = torch.cuda.device_count()
print(f"Number of GPUs detected: {gpu_count}")

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Is CUDA available? False
Number of GPUs detected: 0


## Modelo

In [14]:
import nemo.collections.asr as nemo_asr

In [16]:
model = nemo_asr.models.ASRModel.from_pretrained(MODEL)

[NeMo I 2026-03-20 17:20:09 mixins:208] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2026-03-20 17:20:09 mixins:347] Tokenizer SentencePieceTokenizer initialized with 1152 tokens
[NeMo I 2026-03-20 17:20:09 mixins:347] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-03-20 17:20:09 mixins:347] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-03-20 17:20:09 mixins:347] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-03-20 17:20:09 mixins:347] Tokenizer SentencePieceTokenizer initialized with 1024 tokens
[NeMo I 2026-03-20 17:20:09 aggregate_tokenizer:73] Aggregate vocab size: 5248


[NeMo W 2026-03-20 17:20:09 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 8
    pin_memory: true
    prompt_format: canary2
    max_tps: 25
    max_duration: 40.0
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    bucket_duration_bins:
    - - 3.971
      - 30
    - - 3.971
      - 48
    - - 4.973
      - 37
    - - 4.973
      - 60
    - - 5.85
      - 42
    - - 5.85
      - 71
    - - 6.56
      - 46
    - - 6.56
      - 79
    - - 7.32
      - 49
    - - 7.32
      - 88
    - - 8.19
      - 54
    - - 8.19
      - 99
    - - 8.88
      - 61
    - - 8.88
      - 107
    - - 9.76
      - 66
    - - 9.76
      - 118
    - - 10.56
      - 72
    -

[NeMo I 2026-03-20 17:20:57 save_restore_connector:286] Model EncDecMultiTaskModel was successfully restored from /home/umoqnier/.cache/huggingface/hub/models--nvidia--canary-1b-flash/snapshots/a9a55e0295e7dd50d0c8c2a19491900a0daf24f3/canary-1b-flash.nemo.


## Habilitando soporte de Adapters

In [18]:
model.replace_adapter_compatible_modules()

[NeMo I 2026-03-20 17:20:58 adapter_mixins:169] Swapping class nemo.collections.asr.modules.conformer_encoder.ConformerEncoder with adapter compatible class: nemo.collections.asr.modules.conformer_encoder.ConformerEncoderAdapter
[NeMo I 2026-03-20 17:20:58 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNM with adapter compatible class: nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNMAdapter
[NeMo I 2026-03-20 17:20:58 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoder with adapter compatible class: nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoderAdapter


### Verificación de que *targets* son soportados por el modelo

In [20]:
model.adapter_module_names

['', 'encoder', 'transf_encoder', 'transf_decoder']

### Preparando los *Adapters*

In [21]:
from nemo.collections.common.parts import LinearAdapterConfig

In [22]:
input_dim = model.cfg.encoder.d_model
adapter_dim = 8

In [23]:
enc_adapter_cfg = LinearAdapterConfig(in_features=input_dim, dim=adapter_dim)

### Agregando *Adapters* (solo encoder)

In [24]:
model.add_adapter(name="encoder:enc", cfg=enc_adapter_cfg)

### Congelando parámetros del modelo y descongelando solo los pesos de *Adapters*

In [25]:
model.freeze()
model.unfreeze_enabled_adapters()

[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.0.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.1.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.2.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.3.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.4.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-20 17:21:09 adapter_mixins:466] Froze module encoder.layers.5.conv.batch_norm: BatchNorm1d(102

In [26]:
model.summarize()

  | Name                 | Type                              | Params | Mode
----------------------------------------------------------------------------------
0 | preprocessor         | AudioToMelSpectrogramPreprocessor | 0      | eval
1 | encoder              | ConformerEncoderAdapter           | 811 M  | eval
2 | encoder_decoder_proj | Identity                          | 0      | eval
3 | transf_decoder       | TransformerDecoderNMAdapter       | 72.6 M | eval
4 | log_softmax          | TokenClassifier                   | 5.4 M  | eval
5 | loss                 | SmoothedCrossEntropyLoss          | 0      | eval
6 | spec_augmentation    | SpectrogramAugmentation           | 0      | eval
7 | val_loss             | GlobalAverageLossMetric           | 0      | eval
8 | wer                  | WER                               | 0      | eval
9 | bleu                 | BLEU                              | 0      | eval
----------------------------------------------------------------------

### Comprobando los *Adapters habilidatos*

In [27]:
model.get_enabled_adapters()

['enc']

## Dataset - Mapuche

In [28]:
from huggingface_hub import login

login()

In [29]:
# Do it on terminal
!hf auth login

A new version of huggingface_hub (1.7.2) is available! You are using version 1.6.0.
To update, run: pip install -U huggingface_hub

User is already logged in.


In [30]:
!hf auth whoami

user:  umoqnier
orgs:  ElotlMX,somosnlp-hackathon-2022,somosnlp,UNAMMexico


In [31]:
from datasets import load_dataset, load_dataset_builder

### Exploración del dataset

#### Maputzun

In [17]:
MAP_DATASET_ID = "mengct00/Mapudungun_iwslt26"

In [18]:
map_databuilder = load_dataset_builder(MAP_DATASET_ID)

Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

In [19]:
map_databuilder.info.features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'arn': Value('string'),
 'arn_clean': Value('string'),
 'spa': Value('string')}

In [20]:
map_databuilder.info.splits

{'train': SplitInfo(name='train', num_bytes=24941130380, num_examples=41092, shard_lengths=None, original_shard_lengths=None, dataset_name=None),
 'validation': SplitInfo(name='validation', num_bytes=774427821, num_examples=1201, shard_lengths=None, original_shard_lengths=None, dataset_name=None)}

#### Quechua

In [7]:
import soundfile as sf
import os
from pathlib import Path

In [ ]:
def calculate_total_minutes(root_dir, splits=["train", "valid"]):
    results = {}

    for split in splits:
        wav_dir = Path(root_dir) / split / 'wav'
        total_seconds = 0.0

        if not wav_dir.exists():
            print(f"Directory not found: {wav_dir}")
            continue

        for wav_file in wav_dir.glob('*.wav'):
            try:
                data, samplerate = sf.read(wav_file)
                duration = len(data) / samplerate
                total_seconds += duration
            except Exception as e:
                print(f"Error processing {wav_file}: {e}")

        total_minutes = total_seconds / 60
        results[split] = total_minutes

    return results


data_dir = "data/que_data/que_spa_unconstrained"
durations = calculate_total_minutes(data_dir)

for split, minutes in durations.items():
    print(f"{split.capitalize()} split: {minutes/60:.2f} hours")

Train split: 1.67 hours
Valid split: 1.03 hours


In [6]:
data_dir = "data/que_data/que_spa_synthetic_translation"
durations = calculate_total_minutes(data_dir, splits=["train"])

for split, minutes in durations.items():
    print(f"{split.capitalize()} split: {minutes/60:.2f} hours")

Train split: 8.09 hours


### Preparación del dataset

Canary hace uso de un *prompt* que maneja que tipo de tarea vamos a resolver. Necesitamos entregarle los datos en un formato que cumpla con el formato de la clase que define el *prompt*.

In [27]:
model.prompt?

Type:            Canary2PromptFormatter
String form:     <nemo.collections.common.prompts.canary2.Canary2PromptFormatter object at 0x7f11c42e5490>
File:            ~/devel/master/iwstl-low-resource-experiments/.venv/lib/python3.12/site-packages/nemo/collections/common/prompts/canary2.py
Docstring:       <no docstring>
Class docstring:
:class:`~nemo.collections.common.prompts.formatter.PromptFormatter` is intended to simplify
working with various prompt format templates and encoding them into token ID tensors.

It assumes a dialog-like structure, which is a list of turns, with each turn assigned to a role.
Sub-classes of PromptFormatter define turn templates for each role under TEMPLATE class attribute.
Each template may define some constant parts (e.g. begin-of-turn or end-of-turn tokens, whitespaces, etc.)
and variable parts which we call "slots", that will be provided by the user during training or inference.

A role is typically "user" and "assistant", and some popular models also u

In [28]:
model.prompt.TEMPLATE

{'user': {'template': '<|startofcontext|>|decodercontext|<|startoftranscript|>|emotion||source_lang||target_lang||pnc||itn||timestamp||diarize|',
  'slots': {'decodercontext': nemo.collections.common.prompts.formatter.Text,
   'emotion': Modality.TextLiteral(allowed_values=('<|emo:undefined|>', '<|emo:neutral|>', '<|emo:angry|>', '<|emo:happy|>', '<|emo:sad|>')),
   'source_lang': nemo.collections.common.prompts.formatter.Text,
   'target_lang': nemo.collections.common.prompts.formatter.Text,
   'pnc': Modality.TextLiteral(allowed_values=('True', 'true', 'yes', 'False', 'pnc', 'false', '<|pnc|>', '0', 'nopnc', 'No', '<|nopnc|>', '1', 'no', 'Yes')),
   'itn': Modality.TextLiteral(allowed_values=('True', 'true', 'yes', '<|noitn|>', 'false', 'No', '<|itn|>', '0', 'noitn', 'itn', 'False', '1', 'no', 'Yes')),
   'timestamp': Modality.TextLiteral(allowed_values=('True', 'true', 'yes', 'notimestamp', '<|notimestamp|>', 'false', '<|timestamp|>', '0', 'timestamp', 'No', 'False', '1', 'no', 'Yes

Usaremos la clase `PromptedAudioToTextLhotseDataset` predefinida en la biblioteca de Nemo. Esta clase mapea items del manifest definido por nosotros  a items definidos en el *prompt template* del modelo. Asi, mientras el *manifest* corresponda con los *slots* soportados por el modelo, estos seran manejados por el Dataset automáticamente. 

In [32]:
import torch
from nemo.collections.asr.data.audio_to_text_lhotse_prompted import (
    PromptedAudioToTextMiniBatch,
)
from nemo.collections.common.data.prompt_fn import (
    get_prompt_format_fn,
    #registered_prompt_format_fn,
)
from torch.utils.data import Dataset
from lhotse.dataset import AudioSamples
from lhotse.dataset.collation import collate_vectors
from lhotse import CutSet
from nemo.collections.common.prompts import PromptFormatter
from lhotse.cut import Cut


class MyCanaryPromptedAudioToTextLhotseDataset(Dataset):
    """
    This dataset is based on :class:`~nemo.collections.asr.data.audio_to_text_lhotse.LhotseSpeechToTextBpeDataset`.
    It is a Lhotse-style dataset that converts a mini-batch of Cuts into tensors.
    The main difference from ``LhotseSpeechToTextBpeDataset`` is that we introduce
    a special prompt format for multitask encoder-decoder models.

    To perform the prompt formatting, we accept a ``prompt_format_fn``.
    It's expected to accept:
    * a ``Cut`` a single MonoCut or MixedCut
    * a ``PromptFormatter`` Prepend and append control tokens to the token sequence

    Tokenized utterances will be extended with special prompt tokens according to ``prompt_format_fn`` logic.
    We support cuts with multiple supervision segments -- their tokenized texts will be concatenated before we add the prompt tokens.
    This is useful, for example, in code-switched scenarios where each segment is spoken in a different language.
    """

    def __init__(self, tokenizer: "TokenizerSpec", prompt: PromptFormatter):
        super().__init__()
        self.tokenizer = tokenizer
        self.load_audio = AudioSamples(fault_tolerant=True)
        self.padding_value = self.tokenizer.pad_id
        self.prompt = prompt
        self.prompt_format_fn = get_prompt_format_fn(
            Cut, self.prompt
        )  # Use the default canary prompt function

    def __getitem__(self, cuts: CutSet) -> PromptedAudioToTextMiniBatch:
        audio, audio_lens, cuts = self.load_audio(cuts)
        answers = []
        prompts = []
        prompts_with_answers = []

        for cut in cuts:
            prompted_answers = self.prompt_format_fn(cut, self.prompt)
            answers.append(prompted_answers["answer_ids"])
            prompts.append(prompted_answers["context_ids"])
            prompts_with_answers.append(prompted_answers["input_ids"])

        transcript, transcript_lens = self._collate_tokens(answers)
        prompts_with_answers, prompts_with_answers_lens = self._collate_tokens(
            prompts_with_answers
        )
        prompts, prompt_lens = self._collate_tokens(prompts)

        return PromptedAudioToTextMiniBatch(
            audio=audio,
            audio_lens=audio_lens,
            transcript=transcript,
            transcript_lens=transcript_lens,
            prompt=prompts,
            prompt_lens=prompt_lens,
            prompted_transcript=prompts_with_answers,
            prompted_transcript_lens=prompts_with_answers_lens,
            cuts=cuts.drop_in_memory_data(),
        )

    def _collate_tokens(
        self, tokens: list[list[int] | torch.Tensor]
    ) -> tuple[torch.Tensor, torch.Tensor]:
        tokens = [torch.as_tensor(t) for t in tokens]
        token_lens = torch.tensor([t.size(0) for t in tokens], dtype=torch.long)
        tokens = collate_vectors(tokens, padding_value=self.padding_value)
        return tokens, token_lens

In [37]:
import itertools
import yaml
import tqdm
import lightning.pytorch as L
from omegaconf import OmegaConf
from nemo.collections.asr.parts.utils.manifest_utils import write_manifest, read_manifest
from nemo.collections.common.data.lhotse import get_lhotse_dataloader_from_config
#from nemo.collections.common.prompts import CanaryPromptFormatter

In [41]:
class CanaryMultilingualDataModule(L.LightningDataModule):
    def __init__(
        self, 
        tokenizer, 
        prompt_formatter,
        map_dataset_name: str = "mengct00/Mapudungun_iwslt26",
        que_data_dir: str = "./data/que_data/",
        out_data_dir: str = "./combined_data/", 
        batch_size: int = 8,
        streaming: bool = False,
        max_examples: int | None = None, # Applied per language
        language_mode: str = "multi" # Accepts: "map", "que", "multi"
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_formatter = prompt_formatter
        
        self.map_dataset_name = map_dataset_name
        self.que_data_dir = que_data_dir
        self.out_data_dir = out_data_dir
        self.batch_size = batch_size
        self.streaming = streaming
        self.max_examples = max_examples
        
        if language_mode not in ["map", "que", "multi"]:
            raise ValueError("language_mode must be 'map', 'que', or 'multi'")
        self.language_mode = language_mode

        # AST manifests paths 
        self.train_manifest = os.path.join(out_data_dir, f"train_{self.language_mode}_manifest.json")
        self.val_manifest = os.path.join(out_data_dir, f"val_{self.language_mode}_manifest.json")
        self.test_manifest = os.path.join(out_data_dir, f"test_{self.language_mode}_manifest.json")

    def setup(self, stage=None):
        pass

    def _setup_dataloader(self, config):
        rank = self.trainer.global_rank if self.trainer else 0
        world_size = self.trainer.world_size if self.trainer else 1
        
        return get_lhotse_dataloader_from_config(
            OmegaConf.create(config),
            global_rank=rank,
            world_size=world_size,
            dataset=MyCanaryPromptedAudioToTextLhotseDataset(
                tokenizer=self.tokenizer, 
                prompt=self.prompt_formatter
            ),
        )

    def train_dataloader(self):
        return self._setup_dataloader({
            "manifest_filepath": self.train_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": True,
        })

    def val_dataloader(self):
        return self._setup_dataloader({
            "manifest_filepath": self.val_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": False,
        })

    def test_dataloader(self):
        # Optional: handle cases where test split might not exist
        if not os.path.exists(self.test_manifest):
            print("Warning: No test manifest found, returning val_dataloader for test.")
            return self.val_dataloader()
            
        return self._setup_dataloader({
            "manifest_filepath": self.test_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": False,
        })
    
    def _process_mapudungun(self, split: str) -> list:
        print(f"Processing Mapudungun split: {split}...")
        dataset = load_dataset(self.map_dataset_name, split=split, streaming=self.streaming)
        wav_dir = os.path.join(self.out_data_dir, "map_wavs")
        os.makedirs(wav_dir, exist_ok=True)

        if self.max_examples is not None:
            dataset = dataset.take(self.max_examples) if self.streaming else dataset.select(range(min(len(dataset), self.max_examples)))

        entries = []
        try:
            total = self.max_examples if self.streaming else len(dataset)
        except TypeError:
            total = self.max_examples

        for idx, item in tqdm.tqdm(enumerate(dataset), total=total, desc=f"Mapudungun {split}"):
            audio_info = item.get("audio")
            if not audio_info:
                continue
            
            filepath = os.path.join(wav_dir, f"map_{split}_{idx}.wav")
            # Only write if it doesn't exist to speed up re-runs
            if not os.path.exists(filepath):
                sf.write(filepath, audio_info["array"], audio_info["sampling_rate"])
            duration = len(audio_info["array"]) / audio_info["sampling_rate"]

            text = item.get("spa", "")
            if not text:
                for k, v in item.items():
                    if isinstance(k, str) and (k.endswith("_es") or k.endswith("_spa")) and isinstance(v, str):
                        text = v
                        break
            
            if text:
                entries.append({
                    "audio_filepath": os.path.abspath(filepath),
                    "duration": duration,
                    "text": text,
                    "pnc": "yes",
                    "source_lang": "en", # Placeholder for Mapudungun
                    "target_lang": "es",
                })
        return entries

    def _process_quechua(self, split: str) -> list:
        print(f"Processing Quechua split: {split}...")
        
        # Determine which subdirectories map to this split
        subdirs = []
        if split == "train":
            subdirs = [
                "que_spa_unconstrained/train", 
                "que_spa_synthetic_translation/train"
            ]
        elif split == "validation":
            subdirs = ["que_spa_unconstrained/valid"]
        else:
            return [] # No test split defined locally for Quechua

        entries = []
        for subdir in subdirs:
            split_dir = os.path.join(self.que_data_dir, subdir)
            if not os.path.exists(split_dir):
                print(f"Warning: Quechua directory {split_dir} not found. Skipping.")
                continue

            # Figure out local split name ('train' or 'valid') for the .yaml/.spa files
            local_split = os.path.basename(subdir)
            
            yaml_path = os.path.join(split_dir, "txt", f"{local_split}.yaml")
            text_path = os.path.join(split_dir, "txt", f"{local_split}.spa")
            wav_dir = os.path.join(split_dir, "wav")

            if not os.path.exists(yaml_path) or not os.path.exists(text_path):
                print(f"Warning: Missing metadata/text files in {split_dir}")
                continue

            with open(yaml_path, 'r', encoding='utf-8') as f:
                metadata = yaml.safe_load(f)
            with open(text_path, 'r', encoding='utf-8') as f:
                texts = [line.strip() for line in f.readlines()]

            for i, meta in tqdm.tqdm(enumerate(metadata), total=len(metadata), desc=f"Quechua {subdir}"):
                # Enforce total max_examples across ALL subdirectories for Quechua
                if self.max_examples and len(entries) >= self.max_examples:
                    break
                    
                wav_file = meta['wav']
                duration = meta['duration']
                text = texts[i]
                audio_path = os.path.abspath(os.path.join(wav_dir, wav_file))

                if os.path.exists(audio_path):
                    entries.append({
                        "audio_filepath": audio_path,
                        "duration": duration,
                        "text": text,
                        "pnc": "yes",
                        "source_lang": "fr", # Placeholder for Quechua
                        "target_lang": "es",
                    })
                else:
                    print(f"Warning: Audio file not found: {audio_path}")
            
            # Break outer loop if limit is reached
            if self.max_examples and len(entries) >= self.max_examples:
                break
                
        return entries

    def prepare_data(self):
        if not os.path.exists(self.out_data_dir):
            os.makedirs(self.out_data_dir)

        if not self.max_examples and os.path.exists(self.train_manifest) and os.path.exists(self.val_manifest):
            print(f"Manifests for mode '{self.language_mode}' found. Skipping data preparation.")
            return

        splits = ["train", "validation"] # TODO: Add test when available

        for split in splits:
            print(f"\n--- Preparing {split} split ---")
            
            map_entries = []
            que_entries = []
            
            # Extract based on language_mode flag
            if self.language_mode in ["map", "multi"]:
                map_entries = self._process_mapudungun(split)
            if self.language_mode in ["que", "multi"]:
                que_entries = self._process_quechua(split)
            
            # Interleave entries: 1 Mapudungun, 1 Quechua, 1 Mapudungun...
            combined_entries = []
            for map_entry, que_entry in itertools.zip_longest(map_entries, que_entries):
                if map_entry is not None:
                    combined_entries.append(map_entry)
                if que_entry is not None:
                    combined_entries.append(que_entry)
            
            if combined_entries:
                # Set path based on split name mapping
                if split == "train": 
                    path = self.train_manifest
                elif split == "validation": 
                    path = self.val_manifest
                else: 
                    path = self.test_manifest
                    
                write_manifest(path, combined_entries)
                print(f"Created manifest with {len(combined_entries)} total examples ({len(map_entries)} MAP, {len(que_entries)} QUE): {path}")
            else:
                print(f"No data found for split {split}. Skipping manifest creation.")

        print("Data preparation complete.")

In [49]:
data_loader = CanaryMultilingualDataModule(tokenizer=model.tokenizer, prompt_formatter=model.prompt, streaming=False, batch_size=16, language_mode="multi")

In [ ]:
data_loader.prepare_data()


--- Preparing train split ---
Processing Mapudungun split: train...


Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

data/train-00010-of-00121.parquet:   0%|          | 0.00/221M [00:00<?, ?B/s]

data/train-00011-of-00121.parquet:   0%|          | 0.00/168M [00:00<?, ?B/s]

data/train-00012-of-00121.parquet:   0%|          | 0.00/169M [00:00<?, ?B/s]

...

data/validation-00000-of-00002.parquet:   0%|          | 0.00/405M [00:00<?, ?B/s]

data/validation-00001-of-00002.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/41092 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1201 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/48 [00:00<?, ?it/s]

In [48]:
!head -n 5 {data_loader.train_manifest}

{"audio_filepath": "/home/umoqnier/develop/master/iwstl-low-resource-experiments/combined_data/map_wavs/map_train_0.wav", "duration": 4.172335600907029, "text": "Hola hermano, \u00bfc\u00f3mo te llamas t\u00fa?", "pnc": "yes", "source_lang": "en", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/develop/master/iwstl-low-resource-experiments/data/que_data/que_spa_unconstrained/train/wav/quechua000000.wav", "duration": 1.9941875, "text": "matemos a esos ladrones", "pnc": "yes", "source_lang": "fr", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/develop/master/iwstl-low-resource-experiments/combined_data/map_wavs/map_train_1.wav", "duration": 30.20408163265306, "text": "Hola hermana. Mi nombre Agust\u00edn Curiqueo Quintriqueo se dice mi nombre. Mi tierra se llama Cancura. All\u00ed est\u00e1n mi pap\u00e1, mis mam\u00e1s, mi hermano, mi hermana. El principal de mi familia se llama Jos\u00e9 Sangre Mollfunao, mi viejo es ese pues. As\u00ed es pues, hermana.", "pnc": "yes",

In [36]:
!head -n 5 {data_loader.val_manifest}

{"audio_filepath": "/home/umoqnier/devel/master/iwstl-low-resource-experiments/map_data/wavs/validation_0.wav", "duration": 5.245895691609977, "text": "Pero la machi sola. sola no queda esa.", "pnc": "yes", "task": "ast", "source_lang": "en", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/devel/master/iwstl-low-resource-experiments/map_data/wavs/validation_1.wav", "duration": 15.254512471655328, "text": "Sola, sola no queda pues ya cuando la machi tiene a su ayudante que lo tomo el ayudante que lo tomo pero de arriba bajo eso padre.", "pnc": "yes", "task": "ast", "source_lang": "en", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/devel/master/iwstl-low-resource-experiments/map_data/wavs/validation_2.wav", "duration": 2.6574603174603175, "text": "En la tierra no est\u00e1 su ayudante arriba.", "pnc": "yes", "task": "ast", "source_lang": "en", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/devel/master/iwstl-low-resource-experiments/map_data/wavs/validation_3.w

# Train Model

Ya que han sido preparados los *adapters* es tiempo de entrenar los pesos de los mismos con los datos.

Actualizamos parámetros del optimizador

In [39]:
print(OmegaConf.to_yaml(model.cfg.optim))

name: adamw
lr: 0.0003
betas:
- 0.9
- 0.98
weight_decay: 0.001
sched:
  name: InverseSquareRootAnnealing
  max_steps: 10000
  warmup_steps: 25
  warmup_ratio: null
  min_lr: 1.0e-06



In [40]:
# Setup optimization
model.cfg.optim.lr = 3e-4
model.cfg.optim.sched.warmup_steps = 25

Configuramos un entrenador lighting

In [41]:
trainer = L.Trainer(
    max_steps=200,
    accumulate_grad_batches=1,
    logger=False,
    enable_checkpointing=False,
    check_val_every_n_epoch=5,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [42]:
trainer.fit(model, data_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Manifests found. Skipping data preparation.


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 11.61 GiB of which 35.19 MiB is free. Process 61881 has 56.96 MiB memory in use. Including non-PyTorch memory, this process has 10.23 GiB memory in use. Of the allocated memory 9.90 GiB is allocated by PyTorch, and 29.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
model.save_adapters("mapuche_adapters.pt")

In [ ]:
from torchmetrics.text import SacreBLEUScore

sacrebleu = SacreBLEUScore(n_gram=4)
scores = []
preds = []
gts = []
for pred, gt in zip(ast_preds, ast_gt):
    preds.append(pred)
    gts.append([gt])

# bleu = sum(scores) / len(scores)
sacrebleu.update([p.text for p in preds], gts)
bleu = sacrebleu.compute()
print("BLEU", bleu.item() * 100)